# 09 — GeoLibre demos → a GeoAI discovery assistant

**[🚀 Launch this notebook live](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/lite/lab/index.html?path=09_geolibre_demo_recommender.ipynb)**

The [GeoLibre Official Demos](https://share.geolibre.app/giswqs/collections/official-demos) collection contains 100 interactive projects spanning hazards, climate, mobility, health, energy, biodiversity, and history. This lab treats a curated cross-section of the gallery as a small document corpus. TF–IDF features, clustering, and cosine similarity turn the gallery descriptions into an explainable discovery assistant.

> The compact manifest below was reviewed against the collection on 2026-09-29. It is embedded so the notebook remains reproducible in JupyterLite even if the gallery API changes. Every recommendation links to the live project and its portable `.geolibre.json` file.

In [ ]:
import pandas as pd
from IPython.display import Markdown, display
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

BASE = 'https://share.geolibre.app/giswqs'
DEMOS = [
    ('sister-cities-of-major-world-cities', 'Sister Cities of Major World Cities', 'society economy open data', 'Twin-town and sister-city partnerships drawn as great-circle links, with cities sized by number of relationships.'),
    ('the-global-flight-network', 'The Global Flight Network', 'transport open data', 'Busy airport-to-airport corridors drawn as great circles and weighted by airline service.'),
    ('global-land-cover-2024', 'Global Land Cover 2024', 'nature land imagery open data', 'Global 10 metre land use and land cover derived from Sentinel-2 imagery.'),
    ('arctic-sea-ice-march-vs-september', 'Arctic Sea Ice, March vs September', 'climate imagery swipe open data', 'Winter and summer sea-ice concentration compared with an interactive swipe.'),
    ('global-wildfires-last-7-days', 'Global Wildfires — Last 7 Days', 'natural hazards heatmap open data', 'Recent VIIRS active-fire detections aggregated to cells and weighted by fire radiative power.'),
    ('us-wildfire-perimeters-2025', 'US Wildfire Perimeters 2025', 'natural hazards open data', 'Large US wildfire perimeters from NIFC with area, date, state, county, and cause.'),
    ('global-power-plants-by-fuel', 'Global Power Plants by Fuel', 'energy open data', 'Power plants colored by primary fuel and sized by generating capacity.'),
    ('seasonal-greenness-of-the-earth', 'Seasonal Greenness of the Earth', 'nature land imagery swipe open data', 'January and July MODIS Terra NDVI compared through NASA GIBS.'),
    ('ocean-chlorophyll-from-space', 'Ocean Chlorophyll from Space', 'oceans water imagery open data', 'Satellite chlorophyll-a as a proxy for marine phytoplankton abundance.'),
    ('us-streamflow-right-now', 'US Streamflow Right Now', 'oceans water clusters open data', 'Latest instantaneous streamflow at thousands of USGS streamgages.'),
    ('nasa-global-landslide-catalog', 'NASA Global Landslide Catalog', 'natural hazards open data', 'Reported rainfall-triggered and other landslides with event attributes and locations.'),
    ('manhattan-buildings-through-time', 'Manhattan Buildings Through Time', 'cities infrastructure 3d time slider open data', 'Building footprints extruded by height and colored by construction era.'),
    ('nyc-subway-ridership-2025', 'NYC Subway Ridership 2025', 'transport 3d open data', 'Annual entries at subway station complexes raised as proportional 3D columns.'),
    ('dengue-incidence-1990-2024', 'Dengue Incidence 1990–2024', 'health time slider 3d open data', 'Annual reported dengue incidence by country visualized through time.'),
    ('co2-emissions-per-person-3d', 'CO2 Emissions per Person (3D)', 'climate 3d open data', 'Country fossil carbon dioxide emissions per person, colored and extruded.'),
    ('temperature-anomaly-by-country-1940-2025', 'Temperature Anomaly by Country 1940–2025', 'climate time slider open data', 'Annual surface-air temperature anomaly by country relative to a modern baseline.'),
    ('biodiversity-records-by-country', 'Biodiversity Records by Country', 'nature land open data', 'GBIF species occurrence records counted by country.'),
    ('roads-of-the-roman-empire', 'Roads of the Roman Empire', 'history culture open data', 'Reconstructed Roman road segments with route and distance attributes.'),
]

demos = pd.DataFrame(DEMOS, columns=['slug', 'title', 'tags', 'description'])
demos['project_url'] = BASE + '/' + demos['slug']
demos['json_url'] = demos['project_url'] + '.geolibre.json'
demos[['title', 'tags']]

## Learn themes from the descriptions

TF–IDF gives high weight to terms that distinguish one demo from the others. K-Means then groups projects with similar vocabulary. The labels are descriptive clusters, not authoritative subject classifications.

In [ ]:
corpus = (demos['title'] + ' ' + demos['tags'] + ' ' + demos['description']).tolist()
vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), min_df=1)
X = vectorizer.fit_transform(corpus)

model = KMeans(n_clusters=6, n_init=20, random_state=42)
demos['theme_cluster'] = model.fit_predict(X)

terms = vectorizer.get_feature_names_out()
cluster_terms = {}
for cluster_id, center in enumerate(model.cluster_centers_):
    cluster_terms[cluster_id] = ', '.join(terms[center.argsort()[-6:][::-1]])

summary = (demos.groupby('theme_cluster')['title'].apply(list).to_frame())
summary['distinctive_terms'] = [cluster_terms[i] for i in summary.index]
summary

In [ ]:
# A two-dimensional view of the text-feature space.
import matplotlib.pyplot as plt

xy = TruncatedSVD(n_components=2, random_state=42).fit_transform(X)
fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(xy[:, 0], xy[:, 1], c=demos['theme_cluster'], cmap='tab10', s=80)
for row, (x, y) in zip(demos.itertuples(), xy):
    ax.annotate(row.title, (x, y), xytext=(4, 3), textcoords='offset points', fontsize=8)
ax.set(title='GeoLibre demo similarity map', xlabel='Latent text dimension 1', ylabel='Latent text dimension 2')
ax.grid(alpha=0.2)
plt.show()

## Ask for a workflow

Change `QUERY` to describe a problem, data type, or interaction. Cosine similarity ranks the demos whose descriptions are closest to the request. This is a transparent baseline for a richer semantic-search or retrieval-augmented GeoAI assistant.

In [ ]:
QUERY = 'wildfire hazards with satellite imagery and change detection'
query_vector = vectorizer.transform([QUERY])
scores = cosine_similarity(query_vector, X).ravel()
recommendations = demos.assign(similarity=scores).nlargest(5, 'similarity')

lines = [f'### Recommendations for: “{QUERY}”']
for row in recommendations.itertuples():
    lines.append(
        f'- **[{row.title}]({row.project_url})** — similarity `{row.similarity:.3f}` · '        f'[project JSON]({row.json_url}) · {row.description}'
    )
display(Markdown('\n'.join(lines)))

## What to build next

1. Replace the compact manifest with a versioned gallery export when a stable CORS-enabled catalog endpoint is available.
2. Add structured signals such as layer type, feature count, time-slider fields, and raster/vector source type.
3. Use the returned `.geolibre.json` as an auditable handoff: inspect its layers, extract public data, then search STAC for complementary imagery.
4. Keep a human in the loop. Similar wording does not guarantee that a dataset is suitable, current, licensed for the intended use, or spatially aligned.

## Data & software citations

- [GeoLibre Gallery](https://geolibre.app/gallery/) and [Official Demos collection](https://share.geolibre.app/giswqs/collections/official-demos).
- GeoLibre: https://geolibre.app/ — MIT-licensed software.
- scikit-learn: TF–IDF, K-Means, truncated SVD, and cosine similarity.
- Each live project contains its own description and source attribution; inspect those fields before reuse.